In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='/Users/toszmac/Documents/Pharma_Manufacturing_Data_Analysis_Project/Pharma_Manufacturing_Data_Analysis_Folder/.env')

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


### Question 1: What is the **number of job order** processed per month?

In [3]:
query = """
    SELECT
        d.month,
        d.month_name,
        COUNT(*) AS batch_count
    FROM fact_batch_production f
    INNER JOIN dim_date d ON d.date_id = f.date_id
    WHERE d.year = 2025
        AND f.stage = 'dry_blending'
    GROUP BY d.month, d.month_name
    ORDER BY d.month;
"""

df_monthly_batches = pd.read_sql(query, engine)
print(df_monthly_batches)

    month month_name  batch_count
0       1    January           99
1       2   February          108
2       3      March           69
3       4      April           95
4       5        May           87
5       6       June           53
6       7       July           98
7       8     August          116
8       9  September          120
9      10    October           94
10     11   November           96
11     12   December           29


In [4]:

fig = px.line(
    df_monthly_batches,
    x='month_name',
    y='batch_count',
    title='Monthly Batch Production Volume (2025)',
    height=500,
    width=800,
    labels={
        'month_name': '',
        'batch_count': 'Number of Job Order Processed'
    },
    markers=True
)

fig.update_traces(marker=dict(size=8))

fig.update_layout(
    xaxis={'categoryorder': 'array', 'categoryarray': df_monthly_batches['month_name'].tolist()},
    plot_bgcolor='white',
    showlegend=False
)

fig.show()

### Insight — Monthly Production Volume

Production peaks in **Q3 (August–September)**, coinciding with the 
rainy season when demand for cold, cough, and vitamin products 
typically increases. The June dip reflects a planned 3-week 
operational downtime for audit preparation.

**Recommendation:** Schedule maintenance, machine calibration, 
and employee training to complete by **end of July** to ensure 
full capacity during the Q3 demand surge. If an audit is 
anticipated in mid-2026, preparation should begin by April 
to minimize production impact.

### Question 2: What is the **total units produced** per quarter in 2025?

In [4]:
query_quarterly = """
WITH quarterly_output AS (
    SELECT
        j.jo_id,
        d.quarter,
        CASE
            WHEN p.dosage_form IN ('TABLET', 'BILAYER TABLET')
                THEN MAX(CASE WHEN f.stage = 'compression' 
                         THEN f.actual_output_units END)
            WHEN p.dosage_form = 'CAPSULE'
                THEN MAX(CASE WHEN f.stage = 'encapsulation' 
                         THEN f.actual_output_units END)
            ELSE
                MAX(CASE WHEN f.stage = 'coating' 
                         THEN f.actual_output_units END)
        END AS final_output_units
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    LEFT JOIN dim_date d ON f.date_id = d.date_id
    WHERE d.quarter IS NOT NULL
    GROUP BY j.jo_id, d.quarter, p.dosage_form
)
SELECT
    quarter,
    SUM(final_output_units) AS quarter_output
FROM quarterly_output
WHERE final_output_units IS NOT NULL
GROUP BY quarter
ORDER BY quarter;
"""

df_quarterly = pd.read_sql(query_quarterly, engine)

# Add readable quarter labels
df_quarterly['quarter_label'] = df_quarterly['quarter'].map({
    1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'
})

df_quarterly

,quarter,quarter_output,quarter_label
0,1,69495991.0,Q1
1,2,52593119.0,Q2
2,3,81870674.0,Q3
3,4,76186443.0,Q4


In [6]:
fig = px.bar(
    df_quarterly,
    x='quarter_label',
    y='quarter_output',
    title='Total Units Produced per Quarter — 2025',
    height=500,
    width=800,
    labels={
        'quarter_label': '',
        'quarter_output': 'Total Units Produced'
    },
    text=df_quarterly['quarter_output'].apply(lambda x: f"{x/1e6:.1f}M"),
    color='quarter_output',
    color_continuous_scale='Blues'
)

fig.update_traces(
    textposition='auto',
    insidetextanchor='end'
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis=dict(showticklabels=False, showgrid=False, title='')
)

fig.show()

### Insight - Quarterly Production Output (2025)

Large volume is produced at the last two quarter of the year. 
Q3 gets the highest amount output (81.9m), which is attributed to a high job order demand. 
Q2 gets the lowest output due to least batch count caused by audit preparation and holy week downtime.

**Recommendation**: Pre-build inventory in Q1 to buffer Q2 downtime.
Schedule overtime in March and early April before Holy Week
to absorb the production gap.


In [ ]:
dosage_form_query = """
WITH dosage_form_output AS (
    SELECT
        j.jo_id,
        p.dosage_form,
        CASE
            WHEN p.dosage_form IN ('TABLET', 'BILAYER TABLET')
                THEN MAX(CASE WHEN f.stage = 'compression' 
                         THEN f.actual_output_units END)
            WHEN p.dosage_form = 'CAPSULE'
                THEN MAX(CASE WHEN f.stage = 'encapsulation' 
                         THEN f.actual_output_units END)
            ELSE
                MAX(CASE WHEN f.stage = 'coating' 
                         THEN f.actual_output_units END)
        END AS final_output_units
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    LEFT JOIN dim_date d ON f.date_id = d.date_id
    WHERE d.quarter IS NOT NULL
    GROUP BY j.jo_id, p.dosage_form
)

SELECT
    dosage_form,
    SUM(final_output_units) AS total_output
FROM dosage_form_output
WHERE final_output_units IS NOT NULL
GROUP BY dosage_form
ORDER BY total_output desc;
"""

df_dosage = pd.read_sql(dosage_form_query, engine)

# Add millions label
df_dosage['label'] = df_dosage['total_output'].apply(
    lambda x: f"{x/1e6:.1f}M"
)
df_dosage


,dosage_form,total_output,label
0,FILM-COATED TABLET,143426225.0,143.4M
1,CAPSULE,61045209.0,61.0M
2,SUSTAINED-RELEASE TABLET,43618643.0,43.6M
3,TABLET,29063976.0,29.1M
4,BILAYER TABLET,1832295.0,1.8M
5,EXTENDED RELEASE TABLET,478730.0,0.5M
6,ENTERIC-COATED TABLET,382120.0,0.4M
7,MODIFIED RELEASE TABLET,299029.0,0.3M


In [ ]:
fig = px.bar(
    df_dosage,
    x='total_output',
    y='dosage_form',
    orientation='h',
    title='Total Units Produced by Dosage Form — 2025',
    labels={
        'total_output': 'Total Units Produced',
        'dosage_form': ''
    },
    text='label',
    color='total_output',
    color_continuous_scale='Blues'
)

fig.update_traces(
    textposition='auto',
    insidetextanchor='end'
)


fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showticklabels=False, showgrid=False, title='')
)

fig.show()

### Insight — Output by Dosage Form

Film-coated tablets dominate production at **51.2% of total output**, 
reflecting the facility's core competency in coating technology. 
Capsules (21.8%) and sustained-release tablets (15.7%) round out 
the top three, together accounting for 88.7% of all units produced.

**Recommendations:**
1. Prioritize coating process improvements (cycle time, yield, 
   solution utilization) — impacts the majority of output.
2. During low capsule order periods, consider redeploying 
   encapsulation personnel to tablet or coating lines to 
   maintain labor efficiency.